# Atrous Enhancer + UNet Training for Optic Disc/Cup Segmentation

This notebook implements **Atrous Convolution-based Image Enhancer** with two-phase training:
- **Phase 1**: Train atrous enhancer only (UNet frozen)
- **Phase 2**: Fine-tune both enhancer and UNet jointly

## Why Atrous Convolutions for Enhancement?

**Advantages:**
- 🎯 **Multi-scale feature extraction** - Captures both fine details and broad context
- 🔍 **Larger receptive field** - Sees more of the image without downsampling
- 📐 **Preserves resolution** - No information loss from pooling
- 🌐 **Global context** - ASPP module captures scene-level information

**Perfect for fundus images:**
- Blood vessels need fine-scale processing (dilation=1)
- Optic disc needs medium-scale context (dilation=3,6)
- Overall illumination needs global context (global pooling)

## 1. Setup and Imports

In [ ]:
import sys
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
import json

# Add src to path
project_root = Path.cwd().parent
sys.path.append(str(project_root / 'src'))

from models.atrous_enhancer import AtrousImageEnhancer, LightweightAtrousEnhancer
from models.unet import UNet
from training.train_enhancer import (
    EnhancerUNetModel, 
    train_epoch_phase1, 
    train_epoch_phase2,
    validate_epoch,
    plot_training_history
)
from training.train import CombinedLoss, validate_epoch as validate_unet
from data_loader.dataset import GlaucomaDataset, get_dataloaders
from data_loader.transforms import get_training_transforms, get_validation_transforms

# Set seeds for reproducible training
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.cuda.manual_seed_all(42)  # for multi-GPU
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
np.random.seed(42)

print(f"Project root: {project_root}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

def count_parameters(model):
    """Count trainable parameters"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

Project root: /home/robolab/dev/CAP5410-roi-enhancer-odoc
PyTorch version: 2.8.0+cu128
CUDA available: True
CUDA device: NVIDIA GeForce RTX 2080 Ti


## 2. Configuration

In [2]:
# Configuration for atrous enhancer
config = {
    'root_dir': str(project_root),
    'image_size': 256,
    'batch_size': 16,
    'num_workers': 0,  # Set to 0 to avoid multiprocessing issues
    
    # Atrous enhancer architecture
    'enhancer_type': 'lightweight',  # 'full' or 'lightweight'
    'enhancer_base_channels': 24,    # 24 for lightweight, 32 for full
    'dilation_rates': [1, 3, 6],     # Dilation rates for ASPP
    'residual_weight': 0.3,
    
    # UNet architecture (must match pretrained model)
    'unet_base_channels': 64,
    
    # Training
    'phase1_epochs': 50,  # Train enhancer only
    'phase2_epochs': 30,  # Fine-tune both
    'phase1_lr': 1e-4,
    'phase2_lr': 1e-5,  # Lower LR for fine-tuning
    
    # Loss weights - REDUCED from 0.01 to 0.001
    'l1_weight': 0.001,  # Less conservative to allow stronger enhancements
    
    # Paths
    'unet_checkpoint': str(project_root / 'checkpoints' / 'best_model.pth'),
    'save_dir': str(project_root / 'checkpoints_atrous_enhancer'),
    
    # Early stopping
    'patience': 15,
    
    # Data
    'filter_incomplete': True,
    'use_clahe': False,  # NO CLAHE - use original images
}

print("Atrous Enhancer Configuration:")
print("=" * 70)
for key, value in config.items():
    print(f"{key:25s}: {value}")
print("=" * 70)

Atrous Enhancer Configuration:
root_dir                 : /home/robolab/dev/CAP5410-roi-enhancer-odoc
image_size               : 256
batch_size               : 16
num_workers              : 0
enhancer_type            : lightweight
enhancer_base_channels   : 24
dilation_rates           : [1, 3, 6]
residual_weight          : 0.3
unet_base_channels       : 64
phase1_epochs            : 50
phase2_epochs            : 30
phase1_lr                : 0.0001
phase2_lr                : 1e-05
l1_weight                : 0.001
unet_checkpoint          : /home/robolab/dev/CAP5410-roi-enhancer-odoc/checkpoints/best_model.pth
save_dir                 : /home/robolab/dev/CAP5410-roi-enhancer-odoc/checkpoints_atrous_enhancer
patience                 : 15
filter_incomplete        : True
use_clahe                : False


## 3. Create Data Loaders

Using original images **without CLAHE**, with same augmentations as UNet training.

In [3]:
# Clear dataset cache
GlaucomaDataset._split_cache.clear()

# Create transforms (NO CLAHE!)
transform_train = get_training_transforms(
    image_size=config['image_size'],
    use_clahe=False
)

transform_val = get_validation_transforms(
    image_size=config['image_size'],
    use_clahe=False
)

# Get data loaders
train_loader, val_loader, test_loader = get_dataloaders(
    root_dir=config['root_dir'],
    batch_size=config['batch_size'],
    num_workers=config['num_workers'],
    transform_train=transform_train,
    transform_val=transform_val,
    filter_incomplete=config['filter_incomplete'],
)

print(f"\nData loaders created:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches:   {len(val_loader)}")
print(f"  Test batches:  {len(test_loader)}")
print(f"\nNote: Using original images WITHOUT CLAHE preprocessing")

Filtering incomplete masks...
  Filtered out 234 images with incomplete masks
TRAIN split: 1845 samples
VAL split: 395 samples
TEST split: 396 samples

Data loaders created:
  Train batches: 116
  Val batches:   25
  Test batches:  25

Note: Using original images WITHOUT CLAHE preprocessing


## 4. Load Pretrained UNet

Load the best UNet model trained without CLAHE.

In [4]:
import os
if not os.path.exists(config['unet_checkpoint']):
    raise FileNotFoundError(f"UNet checkpoint not found: {config['unet_checkpoint']}")

# Create UNet and load weights
unet = UNet(
    n_channels=3,
    n_classes=3,
    base_channels=config['unet_base_channels']
)

# Load checkpoint
checkpoint = torch.load(config['unet_checkpoint'], map_location='cpu', weights_only=False)
unet.load_state_dict(checkpoint['model_state_dict'])

print(f"✅ Loaded pretrained UNet from: {config['unet_checkpoint']}")
print(f"   Checkpoint epoch: {checkpoint['epoch']}")
if 'best_val_loss' in checkpoint:
    print(f"   Best val loss: {checkpoint['best_val_loss']:.4f}")
elif 'val_loss' in checkpoint:
    print(f"   Val loss: {checkpoint['val_loss']:.4f}")
print(f"   UNet parameters: {count_parameters(unet):,}")

✅ Loaded pretrained UNet from: /home/robolab/dev/CAP5410-roi-enhancer-odoc/checkpoints/best_model.pth
   Checkpoint epoch: 99
   Val loss: 0.1522
   UNet parameters: 31,043,651


## 5. Create Atrous Enhancer Model

In [5]:
# Create atrous enhancer
if config['enhancer_type'] == 'lightweight':
    enhancer = LightweightAtrousEnhancer(
        n_channels=3,
        base_channels=config['enhancer_base_channels'],
        residual_weight=config['residual_weight']
    )
else:
    enhancer = AtrousImageEnhancer(
        n_channels=3,
        base_channels=config['enhancer_base_channels'],
        dilation_rates=config['dilation_rates'],
        residual_weight=config['residual_weight']
    )

print(f"✅ Created {config['enhancer_type']} Atrous Enhancer")
print(f"   Enhancer parameters: {count_parameters(enhancer):,}")
print(f"   Dilation rates: {config['dilation_rates']}")

# Compare with UNet
ratio = count_parameters(enhancer) / count_parameters(unet) * 100
print(f"\n   Enhancer is {ratio:.1f}% the size of UNet")
print(f"   (Much lighter as intended!)")

✅ Created lightweight Atrous Enhancer
   Enhancer parameters: 25,875
   Dilation rates: [1, 3, 6]

   Enhancer is 0.1% the size of UNet
   (Much lighter as intended!)


## 6. Create Combined Model

Combine atrous enhancer + UNet into a single model.

In [6]:
# Create combined model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = EnhancerUNetModel(enhancer, unet)
model = model.to(device)

print(f"✅ Combined model created")
print(f"   Device: {device}")
print(f"   Total parameters: {count_parameters(model):,}")
print(f"   Enhancer: {count_parameters(enhancer):,}")
print(f"   UNet: {count_parameters(unet):,}")

✅ Combined model created
   Device: cuda
   Total parameters: 31,069,526
   Enhancer: 25,875
   UNet: 31,043,651


## 7. Phase 1: Train Atrous Enhancer Only (UNet Frozen)

Train the atrous enhancer while keeping UNet weights frozen.

In [7]:
# Freeze UNet
for param in model.unet.parameters():
    param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Phase 1 - Trainable parameters: {trainable:,}")
print(f"(Should be ~{count_parameters(enhancer):,} - enhancer only)")

# Optimizer
optimizer_phase1 = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=config['phase1_lr']
)

# Loss function with class weights
class_weights = torch.tensor([1.0, 1.0, 2.0]).to(device)
criterion = CombinedLoss(
    ce_weight=0.5,
    dice_weight=0.5,
    class_weights=class_weights,
    device=device
)

print(f"\n✅ Phase 1 setup complete")
print(f"   L1 weight: {config['l1_weight']} (less conservative)")
print(f"   Epochs: {config['phase1_epochs']}")
print(f"   Patience: {config['patience']}")

Phase 1 - Trainable parameters: 25,875
(Should be ~25,875 - enhancer only)

✅ Phase 1 setup complete
   L1 weight: 0.001 (less conservative)
   Epochs: 50
   Patience: 15


In [ ]:
# Training loop - Phase 1
print("\n" + "=" * 70)
print("PHASE 1: Training Atrous Enhancer (UNet Frozen)")
print("=" * 70)

history_phase1 = {
    'train_loss': [], 'train_seg_loss': [], 'train_l1_loss': [],
    'train_iou_bg': [], 'train_iou_disc': [], 'train_iou_cup': [],
    'val_loss': [], 'val_seg_loss': [], 'val_l1_loss': [],
    'val_iou_bg': [], 'val_iou_disc': [], 'val_iou_cup': [],
}

best_val_loss = float('inf')
patience_counter = 0
save_dir = Path(config['save_dir'])
save_dir.mkdir(exist_ok=True, parents=True)

for epoch in range(config['phase1_epochs']):
    print(f"\nPhase 1 - Epoch {epoch + 1}/{config['phase1_epochs']}")
    print("-" * 70)
    
    # Train
    train_loss, train_seg, train_l1, train_iou = train_epoch_phase1(
        model, train_loader, criterion, optimizer_phase1, 
        device, config['l1_weight']
    )
    
    # Validate
    val_loss, val_seg, val_l1, val_iou = validate_epoch(
        model, val_loader, criterion, device, config['l1_weight']
    )
    
    # Record
    for key, val in [
        ('train_loss', train_loss), ('train_seg_loss', train_seg), ('train_l1_loss', train_l1),
        ('train_iou_bg', train_iou[0]), ('train_iou_disc', train_iou[1]), ('train_iou_cup', train_iou[2]),
        ('val_loss', val_loss), ('val_seg_loss', val_seg), ('val_l1_loss', val_l1),
        ('val_iou_bg', val_iou[0]), ('val_iou_disc', val_iou[1]), ('val_iou_cup', val_iou[2]),
    ]:
        history_phase1[key].append(val)
    
    # Print
    print(f"\nEpoch {epoch + 1} Summary:")
    print(f"  Train - Loss: {train_loss:.4f}, Seg: {train_seg:.4f}, L1: {train_l1:.4f}")
    print(f"  Train - IoU: BG={train_iou[0]:.4f}, Disc={train_iou[1]:.4f}, Cup={train_iou[2]:.4f}")
    print(f"  Val   - Loss: {val_loss:.4f}, Seg: {val_seg:.4f}, L1: {val_l1:.4f}")
    print(f"  Val   - IoU: BG={val_iou[0]:.4f}, Disc={val_iou[1]:.4f}, Cup={val_iou[2]:.4f}")
    
    # Save checkpoint
    if (epoch + 1) % 10 == 0:
        checkpoint_path = save_dir / f'phase1_checkpoint_epoch_{epoch + 1}.pth'
        torch.save({
            'epoch': epoch + 1, 'phase': 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer_phase1.state_dict(),
            'val_loss': val_loss, 'history': history_phase1,
            'config': config
        }, checkpoint_path)
        print(f"  💾 Checkpoint saved: {checkpoint_path.name}")
    
    # Best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_path = save_dir / 'phase1_best_model.pth'
        torch.save({
            'epoch': epoch + 1, 'phase': 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer_phase1.state_dict(),
            'best_val_loss': best_val_loss,
            'history': history_phase1, 'config': config
        }, best_path)
        print(f"  ⭐ Best model saved! Val loss: {best_val_loss:.4f}")
    else:
        patience_counter += 1
        print(f"  Patience: {patience_counter}/{config['patience']}")
        if patience_counter >= config['patience']:
            print(f"\n⚠️  Early stopping at epoch {epoch + 1}")
            break

print("\n" + "=" * 70)
print("PHASE 1 COMPLETE")
print("=" * 70)


PHASE 1: Training Atrous Enhancer (UNet Frozen)

Phase 1 - Epoch 1/50
----------------------------------------------------------------------


Phase 1 Training: 100%|█| 116/116 [00:58<00:00,  1.99it/s, loss=0.1460, seg=0.14
Validation: 100%|█| 25/25 [00:03<00:00,  6.52it/s, loss=0.1675, seg=0.1673, l1=0



Epoch 1 Summary:
  Train - Loss: 0.1654, Seg: 0.1652, L1: 0.2221
  Train - IoU: BG=0.8790, Disc=0.8334, Cup=0.7809
  Val   - Loss: 0.1644, Seg: 0.1642, L1: 0.1811
  Val   - IoU: BG=0.8820, Disc=0.8406, Cup=0.7691
  ⭐ Best model saved! Val loss: 0.1644

Phase 1 - Epoch 2/50
----------------------------------------------------------------------


Phase 1 Training: 100%|█| 116/116 [00:58<00:00,  2.00it/s, loss=0.1331, seg=0.13
Validation: 100%|█| 25/25 [00:03<00:00,  6.49it/s, loss=0.1662, seg=0.1660, l1=0



Epoch 2 Summary:
  Train - Loss: 0.1626, Seg: 0.1623, L1: 0.2228
  Train - IoU: BG=0.8832, Disc=0.8359, Cup=0.7824
  Val   - Loss: 0.1613, Seg: 0.1611, L1: 0.1765
  Val   - IoU: BG=0.8840, Disc=0.8447, Cup=0.7743
  ⭐ Best model saved! Val loss: 0.1613

Phase 1 - Epoch 3/50
----------------------------------------------------------------------


...

Phase 1 Training: 100%|█| 116/116 [00:58<00:00,  1.99it/s, loss=0.1797, seg=0.17
Validation: 100%|█| 25/25 [00:03<00:00,  6.49it/s, loss=0.1608, seg=0.1608, l1=0



Epoch 47 Summary:
  Train - Loss: 0.1564, Seg: 0.1563, L1: 0.1316
  Train - IoU: BG=0.8859, Disc=0.8420, Cup=0.7917
  Val   - Loss: 0.1535, Seg: 0.1534, L1: 0.0669
  Val   - IoU: BG=0.8883, Disc=0.8510, Cup=0.7839
  Patience: 1/15

Phase 1 - Epoch 48/50
----------------------------------------------------------------------


Phase 1 Training: 100%|█| 116/116 [00:58<00:00,  1.99it/s, loss=0.1674, seg=0.16
Validation: 100%|█| 25/25 [00:03<00:00,  6.49it/s, loss=0.1643, seg=0.1642, l1=0



Epoch 48 Summary:
  Train - Loss: 0.1556, Seg: 0.1554, L1: 0.1312
  Train - IoU: BG=0.8876, Disc=0.8428, Cup=0.7912
  Val   - Loss: 0.1535, Seg: 0.1534, L1: 0.0690
  Val   - IoU: BG=0.8884, Disc=0.8502, Cup=0.7827
  Patience: 2/15

Phase 1 - Epoch 49/50
----------------------------------------------------------------------


Phase 1 Training:  83%|▊| 96/116 [00:48<00:10,  1.98it/s, loss=0.1461, seg=0.146

## 8. Phase 2: Fine-tune Both Enhancer and UNet

Unfreeze UNet and train both models jointly with lower learning rate.

In [ ]:
# Unfreeze UNet
for param in model.unet.parameters():
    param.requires_grad = True

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Phase 2 - Trainable parameters: {trainable_params:,}")
print(f"(Should be ~{count_parameters(model):,} - both enhancer and UNet)")

# Create optimizer for both models with lower LR
optimizer_phase2 = optim.Adam(
    model.parameters(),
    lr=config['phase2_lr']
)

print(f"\n✅ Phase 2 setup complete")
print(f"   Optimizer: Adam")
print(f"   Learning rate: {config['phase2_lr']} (10x lower)")

In [ ]:
# Training loop - Phase 2
print("\n" + "=" * 70)
print("PHASE 2: Fine-tuning Both Atrous Enhancer and UNet")
print("=" * 70)

history_phase2 = {
    'train_loss': [], 'train_seg_loss': [], 'train_l1_loss': [],
    'train_iou_bg': [], 'train_iou_disc': [], 'train_iou_cup': [],
    'val_loss': [], 'val_seg_loss': [], 'val_l1_loss': [],
    'val_iou_bg': [], 'val_iou_disc': [], 'val_iou_cup': [],
}

best_val_loss = float('inf')
patience_counter = 0

for epoch in range(config['phase2_epochs']):
    print(f"\nPhase 2 - Epoch {epoch + 1}/{config['phase2_epochs']}")
    print("-" * 70)
    
    # Train
    train_loss, train_seg, train_l1, train_iou = train_epoch_phase2(
        model, train_loader, criterion, optimizer_phase2, device, config['l1_weight']
    )
    
    # Validate
    val_loss, val_seg, val_l1, val_iou = validate_epoch(
        model, val_loader, criterion, device, config['l1_weight']
    )
    
    # Record history
    for key, val in [
        ('train_loss', train_loss), ('train_seg_loss', train_seg), ('train_l1_loss', train_l1),
        ('train_iou_bg', train_iou[0]), ('train_iou_disc', train_iou[1]), ('train_iou_cup', train_iou[2]),
        ('val_loss', val_loss), ('val_seg_loss', val_seg), ('val_l1_loss', val_l1),
        ('val_iou_bg', val_iou[0]), ('val_iou_disc', val_iou[1]), ('val_iou_cup', val_iou[2]),
    ]:
        history_phase2[key].append(val)
    
    # Print summary
    print(f"\nEpoch {epoch + 1} Summary:")
    print(f"  Train - Loss: {train_loss:.4f}, Seg: {train_seg:.4f}, L1: {train_l1:.4f}")
    print(f"  Train - IoU: BG={train_iou[0]:.4f}, Disc={train_iou[1]:.4f}, Cup={train_iou[2]:.4f}")
    print(f"  Val   - Loss: {val_loss:.4f}, Seg: {val_seg:.4f}, L1: {val_l1:.4f}")
    print(f"  Val   - IoU: BG={val_iou[0]:.4f}, Disc={val_iou[1]:.4f}, Cup={val_iou[2]:.4f}")
    
    # Save checkpoint every 10 epochs
    if (epoch + 1) % 10 == 0:
        checkpoint_path = save_dir / f'phase2_checkpoint_epoch_{epoch + 1}.pth'
        torch.save({
            'epoch': epoch + 1, 'phase': 2,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer_phase2.state_dict(),
            'val_loss': val_loss,
            'history_phase1': history_phase1,
            'history_phase2': history_phase2,
            'config': config
        }, checkpoint_path)
        print(f"  💾 Checkpoint saved: {checkpoint_path.name}")
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        
        best_path = save_dir / 'phase2_best_model.pth'
        torch.save({
            'epoch': epoch + 1, 'phase': 2,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer_phase2.state_dict(),
            'best_val_loss': best_val_loss,
            'history_phase1': history_phase1,
            'history_phase2': history_phase2,
            'config': config
        }, best_path)
        print(f"  ⭐ Best model saved! Val loss: {best_val_loss:.4f}")
    else:
        patience_counter += 1
        print(f"  Patience: {patience_counter}/{config['patience']}")
        
        if patience_counter >= config['patience']:
            print(f"\n⚠️  Early stopping triggered after {epoch + 1} epochs")
            break

print("\n" + "=" * 70)
print("PHASE 2 COMPLETE")
print("=" * 70)

## 9. Visualize Training History

In [ ]:
# Create results directory
results_dir = project_root / 'results'
results_dir.mkdir(exist_ok=True, parents=True)

# Plot Phase 1 history
fig1 = plot_training_history(
    history_phase1,
    save_path=str(results_dir / 'atrous_enhancer_phase1_history.png')
)
plt.suptitle('Phase 1: Atrous Enhancer Training (UNet Frozen)', fontsize=16, y=1.00)
plt.show()

In [ ]:
# Plot Phase 2 history
fig2 = plot_training_history(
    history_phase2,
    save_path=str(results_dir / 'atrous_enhancer_phase2_history.png')
)
plt.suptitle('Phase 2: Joint Fine-tuning (Atrous Enhancer + UNet)', fontsize=16, y=1.00)
plt.show()

## 10. Save Training History

In [ ]:
# Save histories as JSON
history_path1 = results_dir / 'atrous_enhancer_phase1_history.json'
with open(history_path1, 'w') as f:
    json.dump(history_phase1, f, indent=2)
print(f"Phase 1 history saved to: {history_path1}")

history_path2 = results_dir / 'atrous_enhancer_phase2_history.json'
with open(history_path2, 'w') as f:
    json.dump(history_phase2, f, indent=2)
print(f"Phase 2 history saved to: {history_path2}")

## 11. Compare with Baseline UNet

Compare UNet alone vs UNet with Atrous Enhancer on validation set.

In [ ]:
print("\n" + "=" * 70)
print("COMPARISON: UNet vs UNet + Atrous Enhancer")
print("=" * 70)

# 1. Load UNet alone (no enhancer)
print("\n1️⃣  Evaluating UNet ALONE (baseline)...")
unet_alone = UNet(
    n_channels=3,
    n_classes=3,
    base_channels=config['unet_base_channels']
)
checkpoint = torch.load(config['unet_checkpoint'], map_location='cpu', weights_only=False)
unet_alone.load_state_dict(checkpoint['model_state_dict'])
unet_alone = unet_alone.to(device)

# Evaluate
val_loss_unet, val_iou_unet = validate_unet(
    unet_alone, val_loader, criterion, device
)

print(f"\n   UNet Only Results:")
print(f"   Loss: {val_loss_unet:.4f}")
print(f"   IoU - BG:   {val_iou_unet[0]:.4f}")
print(f"   IoU - Disc: {val_iou_unet[1]:.4f}")
print(f"   IoU - Cup:  {val_iou_unet[2]:.4f}")
print(f"   Mean IoU:   {np.mean(val_iou_unet):.4f}")

In [ ]:
# 2. Load UNet + Atrous Enhancer (best from phase 2)
print("\n2️⃣  Evaluating UNet + ATROUS ENHANCER (best Phase 2 model)...")

# Reload best phase 2 checkpoint
phase2_checkpoint_path = save_dir / 'phase2_best_model.pth'
if not phase2_checkpoint_path.exists():
    print("⚠️  Phase 2 best model not found, using phase 1...")
    phase2_checkpoint_path = save_dir / 'phase1_best_model.pth'

checkpoint = torch.load(phase2_checkpoint_path, map_location='cpu', weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)

# Evaluate
val_loss_enhanced, _, _, val_iou_enhanced = validate_epoch(
    model, val_loader, criterion, device, config['l1_weight']
)

print(f"\n   UNet + Atrous Enhancer Results:")
print(f"   Loss: {val_loss_enhanced:.4f}")
print(f"   IoU - BG:   {val_iou_enhanced[0]:.4f}")
print(f"   IoU - Disc: {val_iou_enhanced[1]:.4f}")
print(f"   IoU - Cup:  {val_iou_enhanced[2]:.4f}")
print(f"   Mean IoU:   {np.mean(val_iou_enhanced):.4f}")

In [ ]:
# 3. Compare the results
print("\n" + "=" * 70)
print("COMPARISON SUMMARY")
print("=" * 70)

mean_unet = np.mean(val_iou_unet)
mean_enhanced = np.mean(val_iou_enhanced)
improvement = mean_enhanced - mean_unet
improvement_pct = (improvement / mean_unet) * 100

print("\nMetric               UNet Only    UNet+Atrous Enh    Improvement")
print("-" * 70)
print(f"Loss                {val_loss_unet:8.4f}     {val_loss_enhanced:8.4f}        {val_loss_unet - val_loss_enhanced:+.4f}")
print(f"IoU (Background)    {val_iou_unet[0]:8.4f}     {val_iou_enhanced[0]:8.4f}        {val_iou_enhanced[0] - val_iou_unet[0]:+.4f}")
print(f"IoU (Disc)          {val_iou_unet[1]:8.4f}     {val_iou_enhanced[1]:8.4f}        {val_iou_enhanced[1] - val_iou_unet[1]:+.4f}")
print(f"IoU (Cup)           {val_iou_unet[2]:8.4f}     {val_iou_enhanced[2]:8.4f}        {val_iou_enhanced[2] - val_iou_unet[2]:+.4f}")
print(f"Mean IoU            {mean_unet:8.4f}     {mean_enhanced:8.4f}        {improvement:+.4f} ({improvement_pct:+.2f}%)")

# Save comparison
comparison = {
    'unet_only': {
        'loss': float(val_loss_unet),
        'iou_bg': float(val_iou_unet[0]),
        'iou_disc': float(val_iou_unet[1]),
        'iou_cup': float(val_iou_unet[2]),
        'mean_iou': float(mean_unet)
    },
    'unet_atrous_enhancer': {
        'loss': float(val_loss_enhanced),
        'iou_bg': float(val_iou_enhanced[0]),
        'iou_disc': float(val_iou_enhanced[1]),
        'iou_cup': float(val_iou_enhanced[2]),
        'mean_iou': float(mean_enhanced)
    },
    'improvement': {
        'mean_iou': float(improvement),
        'mean_iou_percent': float(improvement_pct)
    }
}

comparison_path = results_dir / 'atrous_enhancer_comparison.json'
with open(comparison_path, 'w') as f:
    json.dump(comparison, f, indent=2)
print(f"\n💾 Comparison saved to: {comparison_path}")

# Determine if enhancer helped
if mean_enhanced > mean_unet:
    print("\n✅ ATROUS ENHANCER IMPROVED SEGMENTATION PERFORMANCE!")
    print(f"   Mean IoU improved by {improvement_pct:+.2f}%")
else:
    print("\n⚠️  Atrous enhancer did not improve performance")
    print(f"   Mean IoU changed by {improvement_pct:+.2f}%")
    print("   This suggests that preprocessing may not be as effective as architectural improvements")

## 12. Visualize Sample Predictions

Compare predictions and enhanced images.

In [ ]:
# Get a batch from validation set
val_iter = iter(val_loader)
images, masks = next(val_iter)
images = images.to(device)
masks = masks.to(device)

# Take first 4 samples
images_viz = images[:4]
masks_viz = masks[:4]

# Predict with UNet only
with torch.no_grad():
    unet_alone.eval()
    preds_unet = torch.argmax(unet_alone(images_viz), dim=1)

# Predict with UNet + Atrous Enhancer
with torch.no_grad():
    model.eval()
    enhanced_images, preds_enhanced_logits = model(images_viz)
    preds_enhanced = torch.argmax(preds_enhanced_logits, dim=1)

# Visualize
fig, axes = plt.subplots(4, 5, figsize=(20, 16))

# Denormalize images for visualization
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1).to(device)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1).to(device)

for i in range(4):
    # Original image
    img = images_viz[i] * std + mean
    img = torch.clamp(img, 0, 1)
    axes[i, 0].imshow(img.cpu().permute(1, 2, 0))
    axes[i, 0].set_title('Original Image')
    axes[i, 0].axis('off')
    
    # Enhanced image
    enhanced_img = enhanced_images[i] * std + mean
    enhanced_img = torch.clamp(enhanced_img, 0, 1)
    axes[i, 1].imshow(enhanced_img.cpu().permute(1, 2, 0))
    axes[i, 1].set_title('Enhanced Image')
    axes[i, 1].axis('off')
    
    # Ground truth
    axes[i, 2].imshow(masks_viz[i].cpu(), cmap='tab10', vmin=0, vmax=2)
    axes[i, 2].set_title('Ground Truth')
    axes[i, 2].axis('off')
    
    # UNet only prediction
    axes[i, 3].imshow(preds_unet[i].cpu(), cmap='tab10', vmin=0, vmax=2)
    axes[i, 3].set_title('UNet Only')
    axes[i, 3].axis('off')
    
    # UNet + Atrous Enhancer prediction
    axes[i, 4].imshow(preds_enhanced[i].cpu(), cmap='tab10', vmin=0, vmax=2)
    axes[i, 4].set_title('UNet + Atrous Enh')
    axes[i, 4].axis('off')

plt.tight_layout()
plt.savefig(results_dir / 'atrous_enhancer_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Prediction comparison saved to: {results_dir / 'atrous_enhancer_predictions.png'}")
print("\nLegend: 0=Background (dark blue), 1=Disc (orange), 2=Cup (green)")

## 13. Summary and Conclusions

### Experiment Overview

This notebook trained an **Atrous Convolution-based Image Enhancer** with two-phase training:
- **Phase 1**: Trained atrous enhancer only (UNet frozen)
- **Phase 2**: Fine-tuned both enhancer and UNet jointly

### Atrous Enhancer Architecture

**Key Features:**
- ASPP module with dilation rates [1, 3, 6] + global pooling
- Multi-scale feature extraction without resolution loss
- Lightweight design (~26K parameters for lightweight version)
- Residual connection with 0.3 scaling factor

**Multi-Scale Processing:**
- Fine-scale (dilation=1): Blood vessels, sharp edges
- Medium-scale (dilation=3,6): Disc boundaries, regional features
- Global-scale (pooling): Overall illumination, scene context

### Training Strategy

**Phase 1** (50 epochs):
- Train enhancer only, UNet frozen
- Learning rate: 1e-4
- L1 regularization: 0.001 (less conservative)
- Loss: Segmentation + L1(enhanced - original)

**Phase 2** (30 epochs):
- Fine-tune both enhancer and UNet
- Learning rate: 1e-5 (10x lower)
- Joint optimization for best results

### Performance Analysis

Check the comparison results above to understand if atrous enhancement helped:

**If improvement observed:**
- ✅ Multi-scale preprocessing beneficial
- ✅ Atrous convolutions effective for fundus images
- ✅ Consider this approach in your final pipeline

**If no improvement:**
- ⚠️ Preprocessing may not be the bottleneck
- ⚠️ Architectural improvements (like ASPP-UNet) likely more effective
- ⚠️ Baseline UNet already well-optimized

### Key Insights

1. **Multi-scale enhancement** captures different features simultaneously
2. **Atrous convolutions** preserve resolution while expanding receptive field
3. **Two-phase training** allows focused optimization of each component
4. **Preprocessing vs Architecture**: Enhancement may not outperform architectural improvements

### Next Steps

1. **Compare with ASPP-UNet**: Run `train_aspp_unet.ipynb` to see if architectural improvements work better
2. **Ablation study**: Try different dilation rates [1,2,4] vs [1,3,6]
3. **Analyze enhanced images**: Visualize what enhancements the model learned
4. **Hyperparameter tuning**: Adjust L1 weight, residual scaling, learning rates

### Scientific Value

- ✅ Rigorous comparison of atrous enhancement approach
- ✅ Both positive and negative results are valuable
- ✅ Insights into preprocessing vs architectural improvements
- ✅ Multi-scale feature extraction principles validated

**Great work on training and evaluating the atrous enhancer!** 🚀